<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/LangStudio/RSI_Recent_Detail_Summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
import yfinance as yf
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

This notebook serves as a framework for backtesting and analyzing simple trading strategies based on the Relative Strength Index (RSI). It includes two main strategies:

1.  **RSI Oversold**: Identifies potential buying opportunities when the RSI drops below a specified oversold threshold.
2.  **RSI Momentum Ignition**: Pinpoints momentum breakouts when the RSI crosses above an overbought threshold, optionally filtered by a Simple Moving Average (SMA) trend.

The notebook fetches historical stock data, applies a 5-day trading lockout to avoid over-trading, calculates the dollar-value profitability for each signal over a 1-5 day forward lookback, and provides a detailed performance summary including total profit, win rate, average profit per trade, and capital at risk metrics for each strategy.

In [21]:
TRADE_CAPITAL = 5000 # Global constant for the capital allocated per trade
LOOKBACK_DAYS = 30

In [22]:
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
OptionVolume_id = '1OGdLINK3zjlx6-lMq86SVq9TkbcglkeI'
OptionVolume = f'https://drive.google.com/uc?export=download&id={OptionVolume_id}'

OptionVolume200_id = '1gcwD510l4GFGNcKsbExR3GvKnDZwCHy4'
OptionVolume200 = f'https://drive.google.com/uc?export=download&id={OptionVolume200_id}'

In [23]:
import pandas as pd
import yfinance as yf
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# =======================
#   SIGNAL PROFITABILITY
# =======================

def calculate_signal_profitability(df, capital_per_trade=TRADE_CAPITAL):
    """
    Calculates dollar-value profitability for unique signals
    with a 1-10 day forward lookback, applying a 5-day trading lockout.
    """
    # 1. De-duplicate signals: 1 position per stock per day max
    trade_df = df.drop_duplicates(subset=['Date', 'Symbol']).copy()
    trade_df['Date'] = pd.to_datetime(trade_df['Date']).dt.normalize() # Normalize to remove time component
    trade_df = trade_df.sort_values(by=['Symbol', 'Date']).reset_index(drop=True)

    # 2. Fetch historical data for all relevant symbols once
    symbols_to_fetch = trade_df['Symbol'].unique()
    # Fetch enough data to cover the forward lookback from the latest signal
    # and also a few days before the earliest signal for the trading calendar
    start_fetch_date = trade_df['Date'].min() - timedelta(days=15) # Extended for 10-day lookback
    end_fetch_date = trade_df['Date'].max() + timedelta(days=15) # Extended for 10-day lookback

    print(f"\u2005\u02da Analyzing profitability for {len(trade_df)} unique signals. Fetching historical data for {len(symbols_to_fetch)} symbols...")

    historical_data = yf.download(list(symbols_to_fetch), start=start_fetch_date.strftime('%Y-%m-%d'),
                                  end=end_fetch_date.strftime('%Y-%m-%d'), progress=False, auto_adjust=True)

    if historical_data.empty:
        print("⚠️ No historical data downloaded for analysis. Returning empty report.")
        return pd.DataFrame()

    close_prices = pd.DataFrame()
    if isinstance(historical_data.columns, pd.MultiIndex):
        close_prices = historical_data['Close']
    elif len(symbols_to_fetch) == 1 and 'Close' in historical_data.columns:
        # Handle single symbol case where yf.download returns a DataFrame with 'Close' column
        close_prices = historical_data[['Close']].rename(columns={'Close': symbols_to_fetch[0]})
    elif len(symbols_to_fetch) == 1 and isinstance(historical_data, pd.Series): # Sometimes yf.download returns a Series for single symbol
        close_prices = historical_data.to_frame(name=symbols_to_fetch[0])
    else:
        print("⚠️ Could not extract 'Close' prices from historical data. Returning empty report.")
        return pd.DataFrame()

    # Ensure close_prices has a DatetimeIndex
    if not isinstance(close_prices.index, pd.DatetimeIndex):
        close_prices.index = pd.to_datetime(close_prices.index)
    close_prices.index = close_prices.index.normalize() # Normalize index
    close_prices = close_prices.sort_index()

    # 3. Implement 5-day trading lockout
    trade_df['is_valid_signal'] = True

    # Use a list to collect valid signal rows for concatenation
    valid_signal_rows = []

    for symbol in symbols_to_fetch:
        symbol_signals = trade_df[trade_df['Symbol'] == symbol].copy()

        if symbol_signals.empty:
            continue

        if symbol not in close_prices.columns:
            print(f"❌ Trading calendar not available for {symbol} to apply lockout. Skipping lockout for this symbol. All its signals will be considered valid.")
            valid_signal_rows.extend(symbol_signals.to_dict('records')) # Add all signals if lockout can't be applied
            continue

        ticker_trading_days = close_prices[symbol].dropna().index.normalize()
        ticker_trading_days = ticker_trading_days.sort_values().drop_duplicates()

        last_valid_signal_calendar_idx = -np.inf # Index in ticker_trading_days

        for i, row in symbol_signals.iterrows():
            current_signal_date = row['Date']

            # Find the position of the current signal date in the symbol's trading calendar
            actual_signal_trading_day = current_signal_date
            if current_signal_date not in ticker_trading_days:
                # If signal falls on a non-trading day, find the next trading day
                next_trading_day_locs = ticker_trading_days[ticker_trading_days > current_signal_date]
                if not next_trading_day_locs.empty:
                    actual_signal_trading_day = next_trading_day_locs.min()
                else:
                    # No future trading days, cannot determine lockout period. Mark as invalid.
                    trade_df.loc[i, 'is_valid_signal'] = False
                    continue # Skip to next signal

            current_calendar_idx = ticker_trading_days.get_loc(actual_signal_trading_day)

            # 5 trading days lockout: means the next signal must be at least 5 trading days *after* the last valid signal date
            # Example: Signal on Day 0. Lockout until Day 5. Next signal must be on Day 5 or later.
            # Calendar indices: 0 (signal), 1, 2, 3, 4 (locked out), 5 (first eligible day)
            # So, `current_calendar_idx` must be >= `last_valid_signal_calendar_idx + 5`
            if current_calendar_idx >= last_valid_signal_calendar_idx + 5:
                # This signal is valid
                trade_df.loc[i, 'is_valid_signal'] = True
                last_valid_signal_calendar_idx = current_calendar_idx
                valid_signal_rows.append(row.to_dict()) # Collect only valid rows
            else:
                # This signal is within the lockout period
                trade_df.loc[i, 'is_valid_signal'] = False

    if not valid_signal_rows:
        print("⚠️ No valid signals after applying lockout. Returning empty report.")
        return pd.DataFrame()

    # Reconstruct the DataFrame from valid signal rows
    trade_df_final = pd.DataFrame(valid_signal_rows)

    # 4. Trade Count Validation
    total_valid_signals = len(trade_df_final)
    if total_valid_signals < 30:
        print(f"⚠️ LOCKOUT ALERT: Valid Trade_Count ({total_valid_signals}) < 30 threshold after 5-day lockout. Results should be treated as anecdotal.")

    # 5. Profit calculation for valid signals
    def get_forward_profit(row):
        symbol = row['Symbol']
        sig_date = row['Date']
        # Ensure Price is a number, handling '$' if present
        entry_price = float(str(row['Price']).replace('$', '')) if isinstance(row['Price'], (str, np.str_)) else row['Price']

        # Get price series for this ticker starting from the signal date
        if symbol not in close_prices.columns:
            return pd.Series([np.nan] * 10, index=[f'Day_{i}_$' for i in range(1, 11)]) # Changed to 10 days

        # Ensure sig_date is in the index of ticker_series, or find the next available
        ticker_series = close_prices[symbol].loc[sig_date:].head(11) # Day 0 to Day 10 (11 prices total)

        # Calculate shares to purchase without exceeding capital_per_trade
        shares = np.floor(capital_per_trade / entry_price) if entry_price != 0 else 0

        profits = {}

        for i in range(1, 11): # Day 1 through Day 10 (changed to 10 days)
            if len(ticker_series) > i: # Check if the i-th trading day exists (i.e., we have enough forward data)
                exit_price = ticker_series.iloc[i]
                profits[f'Day_{i}_$'] = round((exit_price - entry_price) * shares, 2)
            else:
                profits[f'Day_{i}_$'] = np.nan # Not enough data for this forward day
        return pd.Series(profits)

    profit_results = trade_df_final.apply(get_forward_profit, axis=1)
    final_report = pd.concat([trade_df_final, profit_results], axis=1)

    # Sort the final report by Date
    final_report = final_report.sort_values(by='Date').reset_index(drop=True)

    return final_report

In [24]:
import pandas as pd
import numpy as np
from datetime import timedelta

# ========================
# SUMMARIZE TRADES
# ========================

def calculate_exposure_metrics(signals_df, _unused_price_df=None, trade_size=TRADE_CAPITAL):
    """
    Calculates Max and Avg Value Risked for 1D, 2D, ..., 10D windows,
    representing the sum of capital for all open positions on a given day.
    """
    exposure_results = {}
    windows = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] # User requested windows

    # Ensure 'Date' column is datetime and set as index for time series operations
    signals_for_exposure = signals_df.copy()
    signals_for_exposure['Date'] = pd.to_datetime(signals_for_exposure['Date'])
    signals_for_exposure = signals_for_exposure.set_index('Date').sort_index()

    if signals_for_exposure.empty:
        for days in windows:
            exposure_results[f'Max_Risked_{days}D'] = 0
            exposure_results[f'Avg_Risked_{days}D'] = 0
        return exposure_results

    # Ensure 'Price' column is numeric
    signals_for_exposure['Price'] = signals_for_exposure['Price'].astype(str).str.replace('$', '').astype(float)

    # Determine the full date range to ensure a continuous time series for exposure
    min_sig_date = signals_for_exposure.index.min()
    max_sig_date = signals_for_exposure.index.max()
    # The exposure can extend past the last signal date by the maximum window
    full_time_index = pd.date_range(start=min_sig_date, end=max_sig_date + timedelta(days=max(windows) - 1), freq='D')

    # Calculate actual capital invested per signal based on the 'Price' column
    # The 'Price' column in signals_df (profit_df) is already a float after calculate_signal_profitability
    shares_per_signal = np.floor(trade_size / signals_for_exposure['Price'])
    actual_capital_per_signal = shares_per_signal * signals_for_exposure['Price']

    # Create a Series where index is date and value is actual capital added on that date
    daily_capital_inflow = actual_capital_per_signal.groupby(signals_for_exposure.index).sum()

    # Reindex daily_capital_inflow to the full_time_index, filling NaNs with 0 for days without new signals
    daily_capital_inflow = daily_capital_inflow.reindex(full_time_index, fill_value=0)

    for days in windows:
        # Initialize capital_flow_net with a float dtype to prevent FutureWarning
        capital_flow_net = pd.Series(0.0, index=full_time_index, dtype=float)
        # Capital entering on signal date
        capital_flow_net.loc[daily_capital_inflow.index] += daily_capital_inflow.values

        # Capital exiting after 'days' days. Create a temporary DataFrame to track these exits.
        # Only consider dates in daily_capital_inflow that are within the full_time_index
        valid_inflow_dates = daily_capital_inflow.index.intersection(full_time_index)
        temp_exits_df = pd.DataFrame(index=valid_inflow_dates)
        temp_exits_df['capital_to_exit'] = daily_capital_inflow.loc[valid_inflow_dates]
        temp_exits_df['exit_date'] = temp_exits_df.index + pd.Timedelta(days=days)

        # Aggregate outflows by date, as multiple trades might exit on the same day
        daily_capital_outflow = temp_exits_df.groupby('exit_date')['capital_to_exit'].sum()

        # Apply outflows to capital_flow_net, ensuring index alignment with full_time_index
        common_exit_dates = daily_capital_outflow.index.intersection(capital_flow_net.index)
        capital_flow_net.loc[common_exit_dates] -= daily_capital_outflow.loc[common_exit_dates]

        # The cumulative sum of capital_flow_net gives the total capital exposed on each day
        current_exposure_series = capital_flow_net.cumsum()

        # Ensure exposure doesn't go negative due to calculation quirks and clip to 0.
        current_exposure_series = current_exposure_series.clip(lower=0)

        # Calculate Max and Avg from the resulting daily exposure series
        exposure_results[f'Max_Risked_{days}D'] = current_exposure_series.max() if not current_exposure_series.empty else 0
        exposure_results[f'Avg_Risked_{days}D'] = current_exposure_series.mean() if not current_exposure_series.empty else 0

    return exposure_results


def display_strategy_summary(perf_df, algorithm_name, trade_size, lookback_days):
    """
    Architectural Grade Summary: Restores Alpha Drivers and injects Risk Metrics.
    Standardizes on 1, 2, ..., 10-day forward return windows.
    """
    if perf_df.empty:
        print("⚠️ ARCHITECT NOTICE: No signals to analyze.")
        return

    # 1. DATA VALIDATION: Ensure 10-Day exists (Universal Standard)
    windows = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    available_days = [d for d in windows if f'Day_{d}_$' in perf_df.columns]

    # 2. STATISTICAL RIGOR CHECK
    total_signals = len(perf_df)
    print("\n" + "═" * 100)
    print(f"🚀 SENIOR ARCHITECT: ENHANCED PERFORMANCE & RISK AUDIT")
    print(f" Algorithm: {algorithm_name} | Trade Capital: ${trade_size:,.0f} | Lookback Days: {lookback_days}")
    if total_signals < 30:
        print(f"❌ STATISTICAL ALERT: Trade_Count ({total_signals}) < 30 threshold. This is likely due to the 5-day lockout reducing the number of valid signals. Results should be treated as anecdotal.")
    print("-" * 100)

    # Calculate capital exposure metrics once
    # Pass None for the unused price_df parameter to match original signature
    exposure_metrics = calculate_exposure_metrics(perf_df, _unused_price_df=None, trade_size=trade_size)

    # 3. VECTORIZED METRICS (Including Avg/Max at Risk - now capital exposure)
    summary_list = []

    for d in available_days:
        col = f'Day_{d}_$'

        # Retrieve the new capital exposure metrics
        avg_at_risk_capital = exposure_metrics.get(f'Avg_Risked_{d}D', 0)
        max_at_risk_capital = exposure_metrics.get(f'Max_Risked_{d}D', 0)

        total_profit_val = perf_df[col].sum()
        profit_risk_ratio = total_profit_val / max_at_risk_capital if max_at_risk_capital != 0 else 0

        summary_list.append({
            "Window": f"Day {d}",
            "Total Profit": f"${total_profit_val:,.2f}",
            "Win Rate": f"{(perf_df[col] > 0).mean():.1%}",
            "Avg/Trade": f"${perf_df[col].mean():,.2f}",
            "Median": f"${perf_df[col].median():,.2f}",
            "Std Dev": f"${perf_df[col].std():,.2f}",
            "Avg At Risk": f"${avg_at_risk_capital:,.2f}",
            "Max At Risk": f"${max_at_risk_capital:,.2f}",
            "Profit/Risk": f"{profit_risk_ratio:,.2f}"
        })

    print(pd.DataFrame(summary_list).to_string(index=False))

    # 4. RESTORING TOP 5 ALPHA DRIVERS (Based on max available window)
    if available_days:
        best_window_col = f'Day_{max(available_days)}_$'
        print(f"\n🏆 TOP 5 ALPHA DRIVERS ({max(available_days)}-Day Cumulative):")
        top_5 = perf_df.sort_values(by=best_window_col, ascending=False).head(5)
        for _, row in top_5.iterrows():
            print(f" • {row['Symbol']}: ${row[best_window_col]:,.2f}")

    # 5. CRITIQUE ENGINE
    if 'Day_10_$' not in perf_df.columns: # Changed from Day_5_$
        print("\n⚠️ COMPLIANCE WARNING: 10-Day Alpha window missing. Update fetch logic.") # Changed message

    print("═" * 100)

In [25]:
# ==========================================
# ⚙️ RSI - OVERSOLD
# ==========================================
# Format: (Length, Threshold)
RSI_CONFIG = [
    (16, 25),
    (24, 30),
    (14, 25),
    (22, 30),
    (18, 25),
    (22, 25),
    (26, 30)
]
FALLBACK_SYMBOLS = ["TSLA", "NVDA", "AAPL", "AMD", "MSFT", "BTC-USD"]

# ==========================================
# 📈 PRECISION CALCULATION LOGIC
# ==========================================

def calculate_rsi_precision(series, period=14):
    """Matches Yahoo Finance / Wilder's Smoothing Precision"""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # com=period-1 is the mathematical equivalent to Wilder's Smoothing
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def scan_for_triggers():
    """Main scanning engine for recent RSI events"""
    # 1. Load Symbols
    try:
        df_csv = pd.read_csv(OptionVolume200)
        symbol_col = [c for c in df_csv.columns if 'symbol' in c.lower()][0]
        symbols = df_csv[symbol_col].str.strip().unique().tolist()
        print(f"✅ Loaded {len(symbols)} symbols from OptionVolume200")
    except:
        symbols = FALLBACK_SYMBOLS
        print(f"⚠️ OptionVolume200 not found. Using fallback list: {symbols}")

    report_data = []

    # Calculate dates
    # Fetch 300 extra days to allow RSI math to stabilize (Wilder's Warm-up)
    start_fetch = (datetime.now() - timedelta(days=LOOKBACK_DAYS + 300)).strftime('%Y-%m-%d')
    trigger_cutoff = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).date()

    print(f"🔍 Scanning for RSI triggers using {len(RSI_CONFIG)} configurations since {trigger_cutoff}...")

    for symbol in symbols:
        try:
            # Fetch data once per symbol
            df = yf.download(symbol, start=start_fetch, progress=False, auto_adjust=True)
            if df.empty: continue

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            # NEW: Loop through each Length/Threshold pair
            for length, threshold in RSI_CONFIG:
                # Calculate RSI for this specific length
                rsi_col = f'RSI_{length}'
                df[rsi_col] = calculate_rsi_precision(df['Close'], period=length)

                # TRIGGER: Check against the specific threshold for this length
                trigger_col = f'Trigger_{length}'
                df[trigger_col] = (df[rsi_col] < threshold) & (df[rsi_col].shift(1) >= threshold)

                # Filter for the lookback window
                recent_hits = df[df.index.date >= trigger_cutoff]
                recent_hits = recent_hits[recent_hits[trigger_col] == True]

                for date, row in recent_hits.iterrows():
                    report_data.append({
                        "Date": date.strftime('%Y-%m-%d'),
                        "Symbol": symbol,
                        "Price": round(row['Close'], 2),
                        "RSI_Len": length,           # Added to track which pair triggered
                        "RSI_Val": round(row[rsi_col], 2),
                        "Thresh": threshold          # Added for clarity
                    })
        except Exception as e:
            print(f"❌ Error processing {symbol}: {e}")
            continue

    return report_data

# ==========================================
# 📝 EXECUTION & OUTPUT GENERATION
# ==========================================

if __name__ == "__main__":
    triggers = scan_for_triggers()

    print("\n" + "="*70)
    print(f"🚀 RSI MULTI-PAIR REPORT: LAST {LOOKBACK_DAYS} DAYS")
    print(f"Pairs Scanned: {RSI_CONFIG}")
    print("="*70)

    if not triggers:
        print(f"No triggers found in the last {LOOKBACK_DAYS} days.")
    else:
        results_df = pd.DataFrame(triggers)
        # Sort by Date (newest) then Symbol
        results_df = results_df.sort_values(by=["Date", "Symbol"], ascending=[False, True])
        print(results_df.to_string(index=False))

    print("="*70)

✅ Loaded 200 symbols from OptionVolume200
🔍 Scanning for RSI triggers using 7 configurations since 2026-04-29...

🚀 RSI MULTI-PAIR REPORT: LAST 30 DAYS
Pairs Scanned: [(16, 25), (24, 30), (14, 25), (22, 30), (18, 25), (22, 25), (26, 30)]
      Date Symbol  Price  RSI_Len  RSI_Val  Thresh
2026-05-29   SQQQ  38.08       16    24.98      25
2026-05-28   SQQQ  38.47       24    29.45      30
2026-05-28   SQQQ  38.47       14    24.41      25
2026-05-26   SQQQ  39.28       24    29.95      30
2026-05-26   SQQQ  39.28       22    29.05      30
2026-05-15   LULU 119.14       14    24.22      25
2026-05-15   LULU 119.14       22    29.41      30
2026-05-14   SQQQ  41.07       18    24.63      25
2026-05-14   SQQQ  41.07       26    29.72      30
2026-05-13    ABT  83.83       26    29.58      30
2026-05-13   SQQQ  41.99       16    23.84      25
2026-05-13   SQQQ  41.99       24    29.28      30
2026-05-11    ABT  82.56       18    23.32      25
2026-05-11    ABT  82.56       22    24.96      

In [26]:
if 'results_df' in locals() and not results_df.empty:
    profit_df = calculate_signal_profitability(results_df)

    # Display Summary
    cols_to_show = ['Date', 'Symbol', 'Price', 'Trend'] + [f'Day_{i}_$' for i in range(1, 11)]
    # Filter columns that exist (Trend is only in the Momentum Ignition results)
    actual_cols = [c for c in cols_to_show if c in profit_df.columns]

    print("\n" + "💸" * 20)
    print(f"PROFITABILITY REPORT (Max ${TRADE_CAPITAL:,.0f} per Trade)")
    print("💸" * 20)
    print(profit_df[actual_cols].to_string(index=False))
else:
    print("⚠️ No signals found in results_df to analyze.")
    profit_df = pd.DataFrame() # Initialize as empty DataFrame

# --- Execution ---
display_strategy_summary(profit_df, algorithm_name='RSI Oversold', trade_size=TRADE_CAPITAL, lookback_days=LOOKBACK_DAYS)

 ˚ Analyzing profitability for 17 unique signals. Fetching historical data for 5 symbols...
⚠️ LOCKOUT ALERT: Valid Trade_Count (8) < 30 threshold after 5-day lockout. Results should be treated as anecdotal.

💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
PROFITABILITY REPORT (Max $5,000 per Trade)
💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
      Date Symbol  Price  Day_1_$  Day_2_$  Day_3_$  Day_4_$  Day_5_$  Day_6_$  Day_7_$  Day_8_$  Day_9_$  Day_10_$
2026-05-01    LMT 512.77    48.42   -34.56    13.41    -3.24   -56.34    -4.68    74.07    64.53    68.76     29.16
2026-05-04    ABT  87.54   -21.09   -70.68   -30.21  -183.54  -283.86  -181.83  -211.47  -150.48  -174.99     21.09
2026-05-06   SQQQ  45.59    15.26  -329.18  -367.33  -248.52  -392.40  -492.68  -284.49  -223.45  -132.98   -370.60
2026-05-08    MCD 275.75   -20.70   -16.38    -0.90   -14.04    11.52   120.96    90.90    81.36   151.74    117.36
2026-05-11    ABT  82.56   107.40    76.20   140.40   114.60   321.00   375.60   349.20   312.60   291.00    246.60
2026-05-1

In [27]:
# ==========================================
# ⚙️ RSI MOMENTUM
# ==========================================
# Format: (Length, Threshold)
RSI_CONFIG = [
    (10, 80),
    (14, 75)
]
SMA_TREND = 200         # Baseline trend filter
FALLBACK_SYMBOLS = ["TSLA", "NVDA", "AAPL", "AMD", "MSFT", "COIN", "MSTR"]

# ==========================================
# 📈 PRECISION CALCULATION LOGIC
# ==========================================

def calculate_rsi_precision(series, period=14):
    """Matches Wilder's Smoothing Precision (Yahoo Style)"""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # com=period-1 is equivalent to Wilder's alpha=1/period
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def scan_for_momentum_triggers():
    """Identifies 'Momentum Ignition' (RSI crossing ABOVE threshold)"""
    try:
        df_csv = pd.read_csv(OptionVolume)
        symbol_col = [c for c in df_csv.columns if 'symbol' in c.lower()][0]
        symbols = df_csv[symbol_col].str.strip().unique().tolist()
    except:
        symbols = FALLBACK_SYMBOLS

    report_data = []

    # Fetch 350 days to ensure SMA 200 and RSI 10 are both stable
    start_fetch = (datetime.now() - timedelta(days=LOOKBACK_DAYS + 350)).strftime('%Y-%m-%d')
    trigger_cutoff = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).date()

    print(f"🚀 Scanning for Momentum Ignition (RSI configurations: {RSI_CONFIG}) since {trigger_cutoff}...")

    for symbol in symbols:
        try:
            df = yf.download(symbol, start=start_fetch, progress=False, auto_adjust=True)
            if df.empty or len(df) < SMA_TREND: continue

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            df['SMA_200'] = df['Close'].rolling(window=SMA_TREND).mean()
            df['Trend'] = np.where(df['Close'] > df['SMA_200'], "UP", "DOWN")

            # NEW: Evaluate each specific RSI pair
            for length, threshold in RSI_CONFIG:
                rsi_vals = calculate_rsi_precision(df['Close'], period=length)

                # TRIGGER: Today is above threshold, Yesterday was below (Cross Above)
                trigger = (rsi_vals > threshold) & (rsi_vals.shift(1) <= threshold)

                # Filter for the lookback window
                recent_hits = df[df.index.date >= trigger_cutoff].copy()
                recent_hits['Is_Trigger'] = trigger
                hits = recent_hits[recent_hits['Is_Trigger'] == True]

                for date, row in hits.iterrows():
                    report_data.append({
                        "Date": date.strftime('%Y-%m-%d'),
                        "Symbol": symbol,
                        "Price": f"${row['Close']:.2f}",
                        "RSI_Len": length,           # Track which length triggered
                        "RSI_Val": round(rsi_vals.loc[date], 1),
                        "Thresh": threshold,         # Track the target threshold
                        "Trend": row['Trend']
                    })
        except Exception:
            continue

    return report_data

# ==========================================
# 📝 EXECUTION & OUTPUT GENERATION
# ==========================================

if __name__ == "__main__":
    triggers = scan_for_momentum_triggers()

    print("\n" + "="*80)
    print(f"🚀 RSI MOMENTUM POWER ZONE REPORT (LAST {LOOKBACK_DAYS} DAYS)")
    print(f"Pairs Scanned: {RSI_CONFIG}")
    print("="*80)

    if not triggers:
        print(f"No momentum breakouts detected in the last {LOOKBACK_DAYS} days.")
    else:
        max_results_df = pd.DataFrame(triggers)
        # Sort by Date (newest first)
        max_results_df = max_results_df.sort_values(by="Date", ascending=False)

        print(max_results_df.to_string(index=False))

    print("="*80)


🚀 Scanning for Momentum Ignition (RSI configurations: [(10, 80), (14, 75)]) since 2026-04-29...

🚀 RSI MOMENTUM POWER ZONE REPORT (LAST 30 DAYS)
Pairs Scanned: [(10, 80), (14, 75)]
      Date Symbol    Price  RSI_Len  RSI_Val  Thresh Trend
2026-05-29   IONQ   $72.07       14     75.5      75    UP
2026-05-29    IGV  $101.63       10     80.3      80    UP
2026-05-29   CRWD  $731.00       10     85.3      80    UP
2026-05-29   SMCI   $46.09       10     84.5      80    UP
2026-05-29   SMCI   $46.09       14     79.5      75    UP
2026-05-29    IBM  $298.02       14     80.5      75    UP
2026-05-29    IBM  $298.02       10     86.3      80    UP
2026-05-29    IGV  $101.63       14     76.1      75    UP
2026-05-29   PANW  $281.69       10     81.6      80    UP
2026-05-29   PANW  $281.69       14     80.5      75    UP
2026-05-29   CSCO  $120.42       14     75.7      75    UP
2026-05-29   ORCL  $225.76       14     75.1      75    UP
2026-05-28    AMD  $518.09       14     76.7      75

In [28]:
# --- RSI Momentum Summary ---
if 'max_results_df' in locals() and not max_results_df.empty:
    profit_df = calculate_signal_profitability(max_results_df)

    # Display Summary
    cols_to_show = ['Date', 'Symbol', 'Price', 'Trend'] + [f'Day_{i}_$' for i in range(1, 11)]
    # Filter columns that exist (Trend is only in the Momentum Ignition results)
    actual_cols = [c for c in cols_to_show if c in profit_df.columns]

    print("\n" + "💸" * 20)
    print(f"PROFITABILITY REPORT (Max ${TRADE_CAPITAL:,.0f} per Trade)")
    print("💸" * 20)
    print(profit_df[actual_cols].to_string(index=False))
elif 'profit_df' not in locals(): # Added a check to prevent re-execution if profit_df already exists
    print("⚠️ No signals found in results_df to analyze.")
    profit_df = pd.DataFrame() # Initialize as empty DataFrame

# --- Execution ---
display_strategy_summary(profit_df, algorithm_name='RSI Momentum', trade_size=TRADE_CAPITAL, lookback_days=LOOKBACK_DAYS)

 ˚ Analyzing profitability for 92 unique signals. Fetching historical data for 38 symbols...

💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
PROFITABILITY REPORT (Max $5,000 per Trade)
💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
      Date Symbol    Price Trend  Day_1_$  Day_2_$  Day_3_$  Day_4_$  Day_5_$  Day_6_$  Day_7_$  Day_8_$  Day_9_$  Day_10_$
2026-04-29    AMD  $337.11    UP   243.32   328.02    62.02   254.10  1179.92   998.90  1653.12  1703.52  1556.52   1517.46
2026-04-29   QCOM  $156.00    UP   754.56   672.32   396.16   977.60  1170.24  1489.60  2018.88  2608.96  1737.92   1829.44
2026-04-29   AMZN  $263.04    UP    38.38    99.18   171.19   199.69   227.05   154.47   183.16   113.05    52.82    134.71
2026-04-29    STX  $643.30    UP   212.38   585.41   666.68   893.97  1001.84   861.98   975.38  1334.97  1158.50   1218.35
2026-04-29     BE  $287.97    UP   -78.37    43.35    11.39   123.76   -42.50  -498.61  -457.98   -68.85  -123.76     30.43
2026-04-30   MRVL  $165.15    UP    -6.00   -44.70   108.00   210.00  -154.2